# Fooocus + Civitai Manager for Google Colab

Run this notebook with a **GPU runtime**. It creates an isolated Python 3.10 environment, optionally persists models in Google Drive, starts a password-protected Civitai manager, and launches Fooocus.


In [ ]:
#@title 1. Install Fooocus in an isolated Python 3.10 environment
import pathlib, subprocess
REPO = 'https://github.com/JaeTheOP/fooocus-clone.git'
BRANCH = 'agent/civitai-model-manager'
ROOT = pathlib.Path('/content/fooocus-clone')
MAMBA = pathlib.Path('/content/micromamba')
ENV = pathlib.Path('/content/fooocus-env')
if not MAMBA.exists():
    subprocess.run('curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content --strip-components=1 bin/micromamba', shell=True, check=True)
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
if not ENV.exists():
    subprocess.run([str(MAMBA), 'create', '-y', '-p', str(ENV), 'python=3.10', 'pip'], check=True)
PYTHON = str(ENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', 'torch==2.3.1', 'torchvision==0.18.1', '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements_versions.txt')], check=True)
print('Fooocus environment ready:', ENV)


In [ ]:
#@title 2. Optional Google Drive persistence
USE_GOOGLE_DRIVE = True #@param {type:'boolean'}
import pathlib, shutil
ROOT = pathlib.Path('/content/fooocus-clone')
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    persistent = pathlib.Path('/content/drive/MyDrive/Fooocus')
    for folder in ('checkpoints', 'loras', 'vae', 'embeddings'):
        target = persistent / 'models' / folder
        target.mkdir(parents=True, exist_ok=True)
        local = ROOT / 'models' / folder
        if local.is_symlink(): local.unlink()
        elif local.exists():
            for item in local.iterdir():
                destination = target / item.name
                if not destination.exists(): shutil.move(str(item), str(destination))
            shutil.rmtree(local)
        local.parent.mkdir(parents=True, exist_ok=True)
        local.symlink_to(target, target_is_directory=True)
    output_target = persistent / 'outputs'
    output_target.mkdir(parents=True, exist_ok=True)
    output_local = ROOT.parent / 'outputs'
    if output_local.is_symlink(): output_local.unlink()
    elif output_local.exists(): shutil.rmtree(output_local)
    output_local.symlink_to(output_target, target_is_directory=True)
    print('Persistent files:', persistent)
else:
    print('Using temporary Colab storage.')


In [ ]:
#@title 3. Start the password-protected Civitai Manager
import os, pathlib, re, secrets, subprocess, time
ROOT = pathlib.Path('/content/fooocus-clone')
PYTHON = '/content/fooocus-env/bin/python'
LOG = ROOT / 'civitai_manager.log'
username = 'fooocus'
password = secrets.token_urlsafe(12)
env = os.environ.copy()
env.update({'FOOOCUS_CIVITAI_HOST':'0.0.0.0', 'FOOOCUS_CIVITAI_PORT':'7866', 'FOOOCUS_CIVITAI_SHARE':'1', 'FOOOCUS_CIVITAI_USERNAME':username, 'FOOOCUS_CIVITAI_PASSWORD':password})
with open(LOG, 'w') as log:
    manager = subprocess.Popen([PYTHON, 'civitai_manager.py'], cwd=ROOT, env=env, stdout=log, stderr=subprocess.STDOUT)
public_url = None
for _ in range(90):
    time.sleep(1)
    text = LOG.read_text(errors='ignore') if LOG.exists() else ''
    match = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', text)
    if match:
        public_url = match.group(0); break
    if manager.poll() is not None: raise RuntimeError('Manager stopped unexpectedly:\n' + text[-4000:])
print('Civitai Manager:', public_url or 'Still starting; inspect ' + str(LOG))
print('Username:', username)
print('Password:', password)


In [ ]:
#@title 4. Launch Fooocus
PRESET = 'photoreal_civitai' #@param {type:'string'}
import pathlib, subprocess
ROOT = pathlib.Path('/content/fooocus-clone')
command = ['/content/fooocus-env/bin/python', 'entry_with_update.py', '--share', '--always-high-vram']
if PRESET.strip(): command += ['--preset', PRESET.strip()]
print('Open the public gradio.live URL printed below.')
subprocess.run(command, cwd=ROOT, check=True)


## Usage

Open the Civitai Manager link, sign in with the displayed credentials, and install a checkpoint or LoRA. Then open **Advanced → Models** in Fooocus, click **Refresh All Files**, and select the new asset. Google Drive persistence keeps downloaded models across runtime resets.
